In [1]:
import yfinance as yf
import pandas as pd 
import numpy as np

univers = {
    "Maybank": {
        "ticker": "1155.KL",
        "aliases": [
            "Maybank",
            "Malayan Banking"
        ]
    },

    "CIMB": {
        "ticker": "1023.KL",
        "aliases": [
            "CIMB",
            "CIMB Group"
        ]
    },

    "Public Bank": {
        "ticker": "1295.KL",
        "aliases": [
            "Public Bank"
        ]
    },

    "Tenaga Nasional": {
        "ticker": "5347.KL",
        "aliases": [
            "Tenaga Nasional",
            "TNB"
        ]
    },

    "Petronas Chemicals": {
        "ticker": "5183.KL",
        "aliases": [
            "Petronas Chemicals",
            "PETRONAS Chemicals",
            "PCG"
        ]
    }
}

tickers = [c["ticker"] for c in univers.values()]

In [2]:
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://theedgemalaysia.com/"

PAGE_URL = (
    "https://theedgemalaysia.com/"
    "flash-categories/Companies%20in%20the%20news"
)

API_URL = (
    "https://theedgemalaysia.com/"
    "api/loadMoreFlashCategories"
)

headers = {
    "User-Agent": ("Mozilla/5.0")
}

response = requests.get(
    PAGE_URL,
    headers=headers,
    timeout=20
)

print(response.status_code)
print(response.url)
print(len(response.text))

200
https://theedgemalaysia.com/flash-categories/Companies%20in%20the%20news
157038


In [3]:
print(response.text[:500])

<!DOCTYPE html><html><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width"/><title> Companies In The News | The Edge Malaysia</title><script async="" src="https://www.googletagservices.com/tag/js/gpt.js"></script><script>var googletag = window.googletag || {};
        googletag.cmd = googletag.cmd || [];</script><link rel="stylesheet" href="https://myassets.theedgemalaysia.com/css/listing-scriptdms.css "/><script async="" src="https://myassets.theedgemalaysia.com/js/lis


In [4]:
soup = BeautifulSoup(
    response.text,
    "html.parser"
)

print(soup.title)

<title> Companies In The News | The Edge Malaysia</title>


In [5]:
with open("page.html", "w", encoding="utf-8") as f:
    f.write(soup.prettify())


In [6]:
import json

next_data = soup.find(
    "script",
    id="__NEXT_DATA__"
)

data = json.loads(next_data.string)

with open("data.txt", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)


Data = json.loads(next_data.string)

json.loads() signifie essentiellement :

JSON string → objet Python

texte JSON
        ↓
json.loads()
        ↓
objet Python

Ensuite tu as l'opération inverse :

json.dump(data, f, ensure_ascii=False, indent=4)

json.dump() prend un objet Python et l'écrit dans un fichier au format JSON :

objet Python
        ↓
json.dump()
        ↓
fichier JSON

Avec BeautifulSoup :

balise.string renvoie le contenu texte seulement si la balise contient essentiellement un seul nœud texte. Par exemple :

<p>Hello</p> alors 
balise.string

"Hello"

Mais si tu as :

<p>Hello <b>Bilal</b></p>

alors .string peut renvoyer None, parce que le <p> contient plusieurs éléments internes.

Dans ce cas :

balise.get_text() récupère tout le texte contenu dans la balise : "Hello Bilal"

In [7]:
page_props = data["props"]["pageProps"]

initial_articles = page_props["corporateData"]

total_records = page_props["totalRecords"]


In [8]:
import time

all_articles = initial_articles.copy()


offset = len(initial_articles)


while offset < total_records:

    params = {
        "offset": offset,
        "flash": "Companies in the news"
    }

    response = requests.get(
        API_URL,
        params=params,
        headers=headers,
        timeout=20
    )

    response.raise_for_status()

    data = response.json()

    articles = data["results"]


    if not articles:
        print("No more articles returned.")
        break


    all_articles.extend(articles)


    print(
        f"Offset {offset} -> "
        f"{len(articles)} articles received | "
        f"Total collected: {len(all_articles)}/{total_records}"
    )


    # Avancer selon le nombre réellement reçu
    offset += len(articles)


    # Petite pause pour ne pas surcharger le serveur
    time.sleep(0.2)


print("\nFinished.")
print("Expected:", total_records)
print("Collected:", len(all_articles))

Offset 14 -> 10 articles received | Total collected: 24/3469
Offset 24 -> 10 articles received | Total collected: 34/3469
Offset 34 -> 10 articles received | Total collected: 44/3469
Offset 44 -> 10 articles received | Total collected: 54/3469
Offset 54 -> 10 articles received | Total collected: 64/3469
Offset 64 -> 10 articles received | Total collected: 74/3469
Offset 74 -> 10 articles received | Total collected: 84/3469
Offset 84 -> 10 articles received | Total collected: 94/3469
Offset 94 -> 10 articles received | Total collected: 104/3469
Offset 104 -> 10 articles received | Total collected: 114/3469
Offset 114 -> 10 articles received | Total collected: 124/3469
Offset 124 -> 10 articles received | Total collected: 134/3469
Offset 134 -> 10 articles received | Total collected: 144/3469
Offset 144 -> 10 articles received | Total collected: 154/3469
Offset 154 -> 10 articles received | Total collected: 164/3469
Offset 164 -> 10 articles received | Total collected: 174/3469
Offset 17

Requests.get() signifie :

« Envoie une requête HTTP GET à cette adresse. »

Imaginons :

API_URL = "https://example.com/api"

Python va essentiellement construire une URL ressemblant à :

https://example.com/api?offset=10&flash=Companies%20in%20the%20news

Donc:

params=params

permet de transformer ton dictionnaire en paramètres de requête (query parameters).

data = response.json()

prend le JSON reçu de l'API et le transforme automatiquement en objet Python (dict ou list selon la réponse).


In [9]:
unique_articles = {}

for article in all_articles:
    unique_articles[article["nid"]] = article

all_articles = list(
    unique_articles.values()
)

print(
    "Unique articles:",
    len(all_articles)
)

print(all_articles[0])

Unique articles: 3469
{'nid': 817679, 'type': 'article', 'language': 'english', 'category': 'Corporate,Malaysia', 'options': 'Top Stories', 'flash': 'Companies in the news', 'tags': '', 'edited': 'S Kanagaraju', 'title': 'IHH, LSH Capital, Alam Maritim, Bermaz Auto, Paragon Globe, AME Elite, HE Group', 'created': 1789053693000, 'updated': 1789053693000, 'author': 'Choy Nyen Yiau', 'source': 'theedgemalaysia.com', 'audio': 'https://multimedia.theedgemalaysia.com/audios/.6ddcfc10-6b83-4bda-b9af-9320f3b35917.mp3', 'audioflag': 1, 'alias': 'node/817679', 'video_url': '', 'img': 'https://assets.theedgemarkets.com/CITN Companies-in-the-news_theedgemalaysia_35.jpg', 'caption': '', 'summary': 'Here is a brief recap of some corporate announcements that made the news on Thursday'}


In [17]:
import re

def contains_alias(text, alias):

    pattern = rf"(?<!\w){re.escape(alias)}(?!\w)"

    return re.search(
        pattern,
        text,
        flags=re.IGNORECASE
    ) is not None

def starts_with_alias(text, aliases):

    text = text.strip()

    for alias in aliases:

        pattern = rf"^\s*{re.escape(alias)}(?=\W|$)"

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        ):
            return True

    return False

Regex est l'abréviation de Regular Expression, en français expression régulière.

Regex = la règle de recherche

re    = l'outil Python pour utiliser cette règle


In [ ]:
useful_articles = []

for article in all_articles:

    title = article.get("title", "")

    matched_companies = []

    for company, info in univers.items():

        aliases = info["aliases"]

        if any(
            contains_alias(title, alias)
            for alias in aliases
        ):

            matched_companies.append(company)

    if matched_companies:

        article["matched_companies"] = matched_companies

        useful_articles.append(article)

print(len(useful_articles))

713


.get() n'est pas une fonction générale de Python. C'est une méthode des dictionnaires (dict)

Donne-moi la valeur associée à "title". Si "title" n'existe pas, donne-moi "" à la place. 

In [19]:
all_company_blocks = []

for i, article in enumerate(useful_articles):

    print(
        f"Article {i + 1}/{len(useful_articles)}"
    )

    article_url = (
        BASE_URL + article["alias"]
    )

    response = requests.get(
        article_url,
        headers=headers,
        timeout=20
    )

    response.raise_for_status()

    article_soup = BeautifulSoup(
        response.text,
        "lxml"
    )

    infos = article_soup.find_all(
        "div",
        class_="newsTextDataWrapInner"
    )

    # Entreprises trouvées dans le titre
    for company in article["matched_companies"]:

        aliases = univers[company]["aliases"]
        ticker = univers[company]["ticker"]

        for idx, info in enumerate(infos):

            text = info.get_text(
                " ",
                strip=True
            )

            # Le passage doit COMMENCER par l'entreprise
            if starts_with_alias(
                text,
                aliases
            ):

                all_company_blocks.append({
                    "company": company,
                    "ticker": ticker,
                    "text": text,
                    "created": article["created"],
                    "nid": article["nid"],
                    "article_title": article["title"],
                    "article_url": article_url,
                    "info_index": idx
                })

                break

    time.sleep(0.2)

Article 1/713
Article 2/713
Article 3/713
Article 4/713
Article 5/713
Article 6/713
Article 7/713
Article 8/713
Article 9/713
Article 10/713
Article 11/713
Article 12/713
Article 13/713
Article 14/713
Article 15/713
Article 16/713
Article 17/713
Article 18/713
Article 19/713
Article 20/713
Article 21/713
Article 22/713
Article 23/713
Article 24/713
Article 25/713
Article 26/713
Article 27/713
Article 28/713
Article 29/713
Article 30/713
Article 31/713
Article 32/713
Article 33/713
Article 34/713
Article 35/713
Article 36/713
Article 37/713
Article 38/713
Article 39/713
Article 40/713
Article 41/713
Article 42/713
Article 43/713
Article 44/713
Article 45/713
Article 46/713
Article 47/713
Article 48/713
Article 49/713
Article 50/713
Article 51/713
Article 52/713
Article 53/713
Article 54/713
Article 55/713
Article 56/713
Article 57/713
Article 58/713
Article 59/713
Article 60/713
Article 61/713
Article 62/713
Article 63/713
Article 64/713
Article 65/713
Article 66/713
Article 67/713
Arti

In [20]:
df_news = pd.DataFrame(
    all_company_blocks
)

df_news["datetime"] = pd.to_datetime(
    df_news["created"],
    unit="ms",
    utc=True
)

df_news["datetime_my"] = (
    df_news["datetime"]
    .dt.tz_convert("Asia/Kuala_Lumpur")
)

df_news["date"] = (
    df_news["datetime_my"]
    .dt.date
)

In [21]:
print(
    "Total company blocks:",
    len(all_company_blocks)
)

print(
    "\nBlocks by company:"
)

print(
    df_news["company"].value_counts()
)

Total company blocks: 674

Blocks by company:
company
Tenaga Nasional       210
CIMB                  200
Maybank               149
Public Bank            84
Petronas Chemicals     31
Name: count, dtype: int64


In [22]:
represented_articles = df_news["nid"].nunique()

print(
    "\nUseful articles:",
    len(useful_articles)
)

print(
    "Articles represented:",
    represented_articles
)

print(
    "Total company blocks:",
    len(df_news)
)


Useful articles: 713
Articles represented: 598
Total company blocks: 674


In [23]:
represented_pairs = {
    (block["nid"], block["company"])
    for block in all_company_blocks
}

missing_matches = []

for article in useful_articles:

    for company in article["matched_companies"]:

        if (
            article["nid"],
            company
        ) not in represented_pairs:

            missing_matches.append({
                "nid": article["nid"],
                "company": company,
                "title": article["title"],
                "alias": article["alias"]
            })


print(
    "Company matches not found in article body:",
    len(missing_matches)
)

Company matches not found in article body: 135
